In [11]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:


import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

#from matplotlib import pyplot as plt

#import seaborn as sns

#from matplotlib_venn import venn3

import statsmodels.api as sm

#import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import descri_function as des_fun

from pathlib import Path
from datetime import datetime


#

In [2]:
# Read data with model outputs
df = pd.read_parquet('/home/juliane.oliveira/workspace/Data/LR_MV_xgb_mlp_sintetic_08_05_2026.parquet')
#'/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/LR_MV_xgb_mlp_sintetic_08_05_2026.parquet')

df_meta = pd.read_parquet('/home/juliane.oliveira/workspace/Data/sintetic_outbreak_metadata_20260325.parquet')
#/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_outbreak_metadata_20260325.parquet')


# # Create the columns indicating the onset and periods for the sintetic outbreak inserted

# convert to weekly period (ISO-like)
df = df.assign(yw = pd.to_datetime(df['year_week'] + '-1', format='%Y-%W-%w').dt.to_period('W'))
df_meta = df_meta.assign(start_yw = pd.to_datetime(df_meta['start_year_week'] + '-1', format='%Y-%W-%w').dt.to_period('W'))
df_meta = df_meta.assign(end_yw   = pd.to_datetime(df_meta['end_year_week'] + '-1', format='%Y-%W-%w').dt.to_period('W'))



In [3]:
def compute_replicates(df, df_meta):
    df = df.copy()

    for code, set_muni in df.groupby('co_ibge'):
        meta_city = df_meta[df_meta['co_ibge'] == code]

        warning = set_muni['warning_final_mem_surge_01'].values
        yw = set_muni['yw'].values

        for rep, dta in meta_city.groupby('replicate'):

            # --- STEP 1: identify NON-coincident starts ---
            is_start = set_muni['yw'].isin(dta['start_yw']).values
            noncoincident_start = is_start & (warning == 0)

            # store start signal
            df.loc[set_muni.index, f'replicate_{rep}_surge'] = noncoincident_start.astype(int)

            # --- STEP 2: keep only intervals whose start is non-coincident ---
            valid_intervals = dta[dta['start_yw'].isin(set_muni.loc[noncoincident_start, 'yw'])]

            if len(valid_intervals) == 0:
                df.loc[set_muni.index, f'replicate_{rep}_surge_consec'] = 0
                continue

            # build consecutive mask WITHOUT warning condition
            cond_consec = (
                (yw[:, None] >= valid_intervals['start_yw'].values) &
                (yw[:, None] <= valid_intervals['end_yw'].values)
            ).any(axis=1)

            df.loc[set_muni.index, f'replicate_{rep}_surge_consec'] = cond_consec.astype(int)

    return df

In [4]:
result = compute_replicates(df, df_meta)

data = result[result.year_week >= '2024-51']


In [8]:
data.columns.to_list();

In [9]:
# =============================================================================
# MODEL COLLECTION
# =============================================================================

models = {
    'EARS': data.filter(regex=r'^C2_alarms_'),
    'EVI': data.filter(regex=r'^sinal_evi_replicate'),
    'ISF': data.filter(regex=r'^EWS_ISF_replicate'),
    'LOF': data.filter(regex=r'^EWS_LOF_replicate'),
    'OCSVM': data.filter(regex=r'^EWS_OCSVM_replicate'),
    'COPOD': data.filter(regex=r'^EWS_COPOD_replicate'),
    'NGM': data.filter(regex=r'^EWS_Rt_replicate'),
    'MV': data.filter(regex=r'^hard_voting_ivas_rep'),
    'LR': data.filter(regex=r'^sinal_ens_ivas_rep'),
    'XGB': data.filter(regex=r'^signal_ensemble_xgb50_rep'),
    'MLP': data.filter(regex=r'^signal_ensemble_mlp50')
}


In [10]:
# # Prepare data

# ---------------------------------------------------
# Model name cleaner
# ---------------------------------------------------

replace_dict = {
    r'^EWS_COPOD_replicate_\d+$': 'COPOD',
    r'^EWS_ISF_replicate_\d+$': 'ISF',
    r'^EWS_LOF_replicate_\d+$': 'LOF',
    r'^EWS_OCSVM_replicate_\d+$': 'OCSVM',
    r'^EWS_Rt_replicate_\d+$': 'NGM',
    r'^C2_alarms_\d+$': 'EARS',
    r'^sinal_evi_replicate_\d+$': 'EVI',
    r'^signal_ensemble_xgb50_rep_\d+$': 'XGBOOST',
    r'^signal_ensemble_mlp50_rep_\d+$': 'MLP',
    r'^sinal_ens_ivas_rep_\d+$': 'LR',
    r'^hard_voting_ivas_rep_\d+$': 'MV'
}

metrics_keep = ['Sensitivity ',
                'Specificity',
                'PPV', 
                'NPV'
                ]

In [13]:
rep = 0

models = [
        f'C2_alarms_{rep}',
        f'sinal_evi_replicate_{rep}',
        f'EWS_ISF_replicate_{rep}',
        f'EWS_LOF_replicate_{rep}',
        f'EWS_OCSVM_replicate_{rep}',
        f'EWS_COPOD_replicate_{rep}',
        f'EWS_Rt_replicate_{rep}',
        f'hard_voting_ivas_rep_{rep}',
        f'sinal_ens_ivas_rep_{rep}',
        f'signal_ensemble_xgb50_rep_{rep}',
        f'signal_ensemble_mlp50_rep_{rep}'
    ]

warn_col = f'replicate_{rep}_surge'
warn_col_with_consec = f'replicate_{rep}_surge_consec'


df_warning_count = des_fun.antici_count(
            data,
            'C2_alarms_0',
            warn_col,
            warn_col_with_consec,
            'co_ibge'
        )

performance_summary = des_fun.summarize_performance(
            df_warning_count
        )

In [15]:
performance_summary.Metric.unique()

array(['Total Warnings', 'Early Detection (1-3 weeks)',
       'Timely Detection (0 weeks)', 'Missed Warnings', 'Sensitivity ',
       'Specificity', 'PPV', 'NPV', 'POD', 'FPR', 'Precision (%)',
       'Timeliness', 'Score', '3 weeks earlier', '2 weeks earlier',
       '1 week earlier', 'Same week (0)', 'Max_timeliness', 'TP + FN',
       'TN + FP', 'TP_ + FP', 'TN + FN', 'TP', 'FN', 'TN', 'FP', 'TP_',
       'n3', 'n2', 'n1', 'n0'], dtype=object)

In [16]:
dta3 = performance_summary[
        performance_summary.Metric.isin(metrics_keep)
    ].copy()

In [17]:
dta3

,Metric,Value
4,Sensitivity,84.2%
5,Specificity,89.6%
6,PPV,82.7%
7,NPV,99.2%


In [12]:
# ---------------------------------------------------
# Main loop
# ---------------------------------------------------

# =============================================================================
# OUTPUT DIRECTORY
# =============================================================================

today = datetime.now().strftime("%d_%m_%Y")

out_dir = Path(
    f"/home/juliane.oliveira/workspace/Data/timing_metrics"
)

out_dir.mkdir(parents=True, exist_ok=True)

for rep in range(1): #32

    print(f"\n===== Processing replicate {rep} =====")

    models = [
        f'C2_alarms_{rep}',
        f'sinal_evi_replicate_{rep}',
        f'EWS_ISF_replicate_{rep}',
        f'EWS_LOF_replicate_{rep}',
        f'EWS_OCSVM_replicate_{rep}',
        f'EWS_COPOD_replicate_{rep}',
        f'EWS_Rt_replicate_{rep}',
        f'hard_voting_ivas_rep_{rep}',
        f'sinal_ens_ivas_rep_{rep}',
        f'signal_ensemble_xgb50_rep_{rep}',
        f'signal_ensemble_mlp50_rep_{rep}'
    ]

    warn_col = f'replicate_{rep}_surge'
    warn_col_with_consec = f'replicate_{rep}_surge_consec'

    lst1 = []
    lst2 = []

    # ---------------------------------------------------
    # Run models
    # ---------------------------------------------------

    for sel_model in models:

        print(f'Running model: {sel_model}')

        # Skip missing columns safely
        if sel_model not in data.columns:
            print(f'Skipping missing model: {sel_model}')
            continue

        df_warning_count = des_fun.antici_count(
            data,
            sel_model,
            warn_col,
            warn_col_with_consec,
            'co_ibge'
        )

        performance_summary = des_fun.summarize_performance(
            df_warning_count
        )

        performance_summary['model'] = sel_model
        df_warning_count['model'] = sel_model

        lst1.append(performance_summary)
        lst2.append(df_warning_count)

    print(lst1)
    # ---------------------------------------------------
    # Combine results
    # ---------------------------------------------------

    summary = pd.concat(lst1, ignore_index=True)

    dta_tab1 = pd.concat(lst2, ignore_index=True)

    # ---------------------------------------------------
    # Clean model names
    # ---------------------------------------------------

    dta_tab1['model'] = dta_tab1['model'].replace(
        replace_dict,
        regex=True
    )

    # ---------------------------------------------------
    # Timing summary
    # ---------------------------------------------------

    dta_tab2 = (
        dta_tab1
        .groupby('model')[
            ['n3', 'n2', 'n1', 'n0', 'missed', 'total_aih_warning']
        ]
        .sum()
        .reset_index()
    )

    # ---------------------------------------------------
    # Metrics
    # ---------------------------------------------------

    dta3 = summary[
        summary.Metric.isin(metrics_keep)
    ].copy()

    dta3['Value'] = (
        dta3['Value']
        .str.replace('%', '', regex=False)
        .astype(float)
        / 100
    )

    dta3['model'] = dta3['model'].replace(
        replace_dict,
        regex=True
    )

    # ---------------------------------------------------
    # Add replicate info
    # ---------------------------------------------------

    dta_tab2['rep'] = rep
    dta3['rep'] = rep

    



===== Processing replicate 0 =====
Running model: C2_alarms_0
Running model: sinal_evi_replicate_0
Running model: EWS_ISF_replicate_0
Running model: EWS_LOF_replicate_0
Running model: EWS_OCSVM_replicate_0


KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------
    # Save replicate results immediately
    # ---------------------------------------------------

    timing_file = out_dir / f"timing_rep_{rep}.parquet"

    metrics_file = out_dir / f"metrics_rep_{rep}.parquet"

    dta_tab2.to_parquet(
        timing_file,
        index=False
    )

    dta3.to_parquet(
        metrics_file,
        index=False
    )

    print(f"Saved replicate {rep}")
